# Assignment 5: Text Generation using RNN, LSTM, and GRU

**Student:** Aryan  
**Objective:** Train and compare three recurrent neural-network architectures for next-word prediction and text generation.

This completed notebook implements all five required customizations:

1. A custom paragraph corpus
2. Embedding dimension increased from 32 to **64**
3. Training increased from 100 to **200 epochs**
4. Hidden units increased from 64 to **128**
5. Generated continuation increased from 5 to **10 words**

## 1. What this project does

A text-generation model learns to predict the next word from the words that came before it. We:

1. Convert words into integer tokens.
2. Create progressively longer n-gram training sequences.
3. Pad every sequence to the same length.
4. Separate each sequence into input words (**X**) and its expected next word (**y**).
5. Train SimpleRNN, LSTM, and GRU models on the same data.
6. Repeatedly predict the next word to generate new text.

Because every model uses the same corpus, preprocessing, embedding size, hidden size, optimizer, and epoch count, the comparison is fair.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Reproducibility: the models start from repeatable random states.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 2. Custom text corpus

The original boilerplate corpus has been replaced with a custom paragraph about space exploration and intelligent machines. Multiple related sentences give the models repeated language patterns to learn.

In [ ]:
corpus = """
space exploration inspires people to imagine new worlds
scientists build intelligent machines to study distant planets
robots can travel through dangerous regions of space
artificial intelligence helps robots understand their surroundings
astronauts depend on reliable technology during every mission
deep learning allows machines to recognize complex patterns
future spacecraft may use intelligent systems to make decisions
human creativity and machine intelligence can solve difficult problems
exploration teaches humanity how to survive beyond earth
careful research makes every space mission safer and smarter
intelligent robots may help astronauts build homes on distant planets
new discoveries encourage scientists to ask deeper questions
"""

# Basic cleaning: remove empty lines and normalize whitespace.
clean_lines = [" ".join(line.lower().split()) for line in corpus.splitlines() if line.strip()]
clean_corpus = "\n".join(clean_lines)

print(clean_corpus)
print("\nNumber of sentences:", len(clean_lines))

## 3. Tokenization and n-gram sequence creation

The tokenizer creates a vocabulary such as `space → 1` and `intelligent → 2`.  
For a sentence like “space exploration inspires people”, progressive sequences are:

- space exploration
- space exploration inspires
- space exploration inspires people

The final word in each sequence becomes the target. The preceding words become the input.

In [ ]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts([clean_corpus])

total_words = len(tokenizer.word_index) + 1
input_sequences = []

for line in clean_lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[: i + 1])

max_len = max(len(sequence) for sequence in input_sequences)
input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding="pre",
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Vocabulary size:", total_words)
print("Longest sequence length:", max_len)
print("Number of training examples:", len(input_sequences))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst padded input:", X[0])
print("First target token:", y[0])

## 4. Shared experiment settings

- **Embedding dimension = 64:** each word is represented by a learned vector of 64 values.
- **Hidden units = 128:** each recurrent layer has more capacity than the original 64-unit model.
- **Epochs = 200:** every model sees the entire training set 200 times.
- **Adam optimizer:** used identically for all three models.
- **Sparse categorical cross-entropy:** suitable because each target is one integer word ID.

In [ ]:
EMBEDDING_DIM = 64       # Required customization: increased from 32
HIDDEN_UNITS = 128        # Required customization: increased from 64
EPOCHS = 200              # Required customization: increased from 100
WORDS_TO_GENERATE = 10    # Required customization: increased from 5

def compile_model(recurrent_layer):
    model = Sequential([
        tf.keras.Input(shape=(max_len - 1,)),
        Embedding(input_dim=total_words, output_dim=EMBEDDING_DIM, mask_zero=True),
        recurrent_layer,
        Dense(total_words, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 5. Model 1 — Vanilla SimpleRNN

SimpleRNN passes a hidden state from one time step to the next. It is a useful baseline, but information from early words can fade during long sequences because of the vanishing-gradient problem.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
rnn_model = compile_model(SimpleRNN(HIDDEN_UNITS))
rnn_model.summary()

rnn_history = rnn_model.fit(
    X,
    y,
    epochs=EPOCHS,
    verbose=0,
)
print("Vanilla RNN training completed.")
print("Final RNN loss:", round(rnn_history.history["loss"][-1], 4))

## 6. Model 2 — LSTM

LSTM adds a cell state and input, forget, and output gates. These gates decide what information to store, discard, and expose, helping the network preserve useful context for longer periods.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
lstm_model = compile_model(LSTM(HIDDEN_UNITS))
lstm_model.summary()

lstm_history = lstm_model.fit(
    X,
    y,
    epochs=EPOCHS,
    verbose=0,
)
print("LSTM training completed.")
print("Final LSTM loss:", round(lstm_history.history["loss"][-1], 4))

## 7. Model 3 — GRU

GRU combines memory control into update and reset gates. It usually has fewer parameters than an equivalent LSTM, so it can train faster while still learning longer dependencies better than a basic RNN.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
gru_model = compile_model(GRU(HIDDEN_UNITS))
gru_model.summary()

gru_history = gru_model.fit(
    X,
    y,
    epochs=EPOCHS,
    verbose=0,
)
print("GRU training completed.")
print("Final GRU loss:", round(gru_history.history["loss"][-1], 4))

## 8. Compare training loss

Cross-entropy loss measures how wrong each next-word probability distribution is. A lower curve means the model is becoming more confident about the correct next words. The dashed marker at epoch 100 shows the original training requirement; training continues to epoch 200 for the student customization.

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(rnn_history.history["loss"], label="SimpleRNN", linewidth=2)
plt.plot(lstm_history.history["loss"], label="LSTM", linewidth=2)
plt.plot(gru_history.history["loss"], label="GRU", linewidth=2)
plt.axvline(99, color="gray", linestyle="--", alpha=0.7, label="Original 100 epochs")
plt.xlabel("Epoch")
plt.ylabel("Sparse categorical cross-entropy loss")
plt.title("Training Loss Comparison: SimpleRNN vs LSTM vs GRU")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Numerical model comparison

Final loss and training accuracy summarize how well each model learned this small corpus. Parameter count indicates model complexity. These are training measurements, not proof that a model will generalize to unseen writing.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["SimpleRNN", "LSTM", "GRU"],
    "Final Loss": [
        rnn_history.history["loss"][-1],
        lstm_history.history["loss"][-1],
        gru_history.history["loss"][-1],
    ],
    "Final Training Accuracy": [
        rnn_history.history["accuracy"][-1],
        lstm_history.history["accuracy"][-1],
        gru_history.history["accuracy"][-1],
    ],
    "Trainable Parameters": [
        rnn_model.count_params(),
        lstm_model.count_params(),
        gru_model.count_params(),
    ],
})

comparison.round({
    "Final Loss": 4,
    "Final Training Accuracy": 4,
})

## 10. Text-generation function

For each new word:

1. Tokenize the current seed text.
2. Pad it to the model's input length.
3. Predict probabilities for every vocabulary word.
4. Use `np.argmax` to select the most probable word.
5. Append that word and repeat.

This is greedy decoding: it is deterministic and easy to understand, although it may repeat common patterns.

In [ ]:
index_to_word = {index: word for word, index in tokenizer.word_index.items()}

def generate_text(model, seed_text, next_words=WORDS_TO_GENERATE):
    generated_text = seed_text.lower().strip()

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([generated_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_len - 1,
            padding="pre",
        )

        probability_array = model.predict(token_list, verbose=0)[0]
        predicted_index = int(np.argmax(probability_array))
        output_word = index_to_word.get(predicted_index)

        # Index 0 is padding and has no word. Stop safely if it is selected.
        if output_word is None or output_word == "<OOV>":
            break

        generated_text += " " + output_word

    return generated_text

## 11. Generate 10 words from a shared seed phrase

All three models start from the exact same seed, **“space exploration”**, so their continuations can be compared directly.

In [ ]:
seed_phrase = "space exploration"

generated_samples = pd.DataFrame({
    "Model": ["SimpleRNN", "LSTM", "GRU"],
    "Generated Text": [
        generate_text(rnn_model, seed_phrase, WORDS_TO_GENERATE),
        generate_text(lstm_model, seed_phrase, WORDS_TO_GENERATE),
        generate_text(gru_model, seed_phrase, WORDS_TO_GENERATE),
    ],
})

for row in generated_samples.itertuples(index=False):
    print(f"{row.Model:9s}: {row._1}")

## 12. Observations and conclusion

### What we should observe

- All three loss curves should generally move downward, showing that the models are learning next-word patterns.
- **SimpleRNN** is the simplest baseline and can learn short local patterns, but its memory is less reliable for long dependencies.
- **LSTM** has the most elaborate gating mechanism and usually has the largest parameter count.
- **GRU** uses fewer gates than LSTM and often offers a useful balance between memory and computational cost.
- The generated text may sound repetitive because the corpus is intentionally small and `np.argmax` always chooses the single most likely word.

### Key limitation

A low training loss on a tiny corpus can mean memorization rather than broad language understanding. Better generation would require a much larger corpus, a validation split, regularization, and probabilistic sampling such as temperature-based decoding.

### Final conclusion

The assignment demonstrates the complete text-generation pipeline: tokenization, n-gram construction, padding, recurrent-model training, loss comparison, and autoregressive next-word generation. All five requested beginner customizations are implemented in executable code.

## Assignment 5 completion checklist

- [x] Custom paragraph corpus used
- [x] Embedding dimension increased to 64
- [x] Training increased to 200 epochs
- [x] Recurrent hidden layers widened to 128 units
- [x] Ten words generated from a shared seed phrase
- [x] SimpleRNN, LSTM, and GRU trained with identical optimizer settings
- [x] Cross-entropy loss comparison chart included
- [x] `np.argmax` used for next-word selection
- [x] Explanations and observations included